# PyTensor and Numba in JupyterLite

This example is adapted from PyTensor's introduction notebook and runs with the Numba linker entirely in the browser.

In [ ]:
import sys
from importlib.metadata import version

import numba
import numpy as np

import pytensor
import pytensor.tensor as pt


assert sys.platform == "emscripten"
{
    "Platform": sys.platform,
    "PyTensor": version("pytensor"),
    "Numba": numba.__version__,
    "NumPy": np.__version__,
}

In [ ]:
x = pt.vector("x", shape=(None,))
z = pt.exp(pt.sin(x))
out = pt.cos((z[None, :] @ z[:, None]).squeeze())

out.dprint()

In [ ]:
numba_fn = pytensor.function([x], out, mode="NUMBA")
numba_fn.dprint(print_destroy_map=True)

In [ ]:
values = np.array([0.25, -0.5, 1.0])
result = numba_fn(values)
expected_z = np.exp(np.sin(values))
expected = np.cos(expected_z @ expected_z)

np.testing.assert_allclose(result, expected)
{"Result": result, "Expected": expected}

In [ ]:
# Reuse the compiled function with a different vector length.
values = np.linspace(-1.0, 1.0, 5)
result = numba_fn(values)
expected_z = np.exp(np.sin(values))
np.testing.assert_allclose(result, np.cos(expected_z @ expected_z))
result